## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets 

In [4]:
## Parameter 
start_date = '20231002'
end_date = '20231015'

start_date_range = datetime.strptime(start_date, '%Y%m%d')
end_date_range = datetime.strptime(end_date, '%Y%m%d')

##base

base = f"""

    with orders_tbl as 
        (
            SELECT   
                order_id,
                service_obj_service_name service_name, 
                captain_id,
                lifetimerides,
                CASE 
                WHEN lifetimerides BETWEEN 1 AND 5 THEN 'a. 1 to 5'
                WHEN lifetimerides BETWEEN 6 AND 10 THEN 'b. 6 to 10'
                WHEN lifetimerides BETWEEN 11 AND 20 THEN 'c. 11 to 20'
                WHEN lifetimerides BETWEEN 21 AND 50 THEN 'd. 21 to 50'
                WHEN lifetimerides > 50 AND lifetimerides <= 100 THEN 'e. 51 to 100'
                WHEN lifetimerides > 100 AND lifetimerides <= 200 THEN 'f. 101 to 200'
                WHEN lifetimerides > 200 AND lifetimerides <= 400 THEN 'g. 201 to 400'
                WHEN lifetimerides > 400 AND lifetimerides <= 800 THEN 'h. 401 to 800'
                WHEN lifetimerides > 800 THEN 'i. 800+'
                END captain_ltr,
                avgcaptainrating
            FROM
                orders.order_logs_snapshot
            
            INNER JOIN datasets.captain_single_view
            on captain_id = captainid 
            
            WHERE
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
    
                AND order_status in ('dropped')
                AND spd_fraud_flag != True
                AND service_obj_service_name IN ('Link', 'Auto')
        ) 
        
        SELECT * FROM orders_tbl
"""

df_base = pd.read_sql(base, connection)
df_base.to_csv('df_base_with_ltr_{}_{}.csv'.format(start_date, end_date), index=False)
df_base

In [5]:
df_base = pd.read_csv('df_base_with_ltr_{}_{}.csv'.format(start_date, end_date)) 
df_base.head(2)

,order_id,service_name,captain_id,lifetimerides,captain_ltr,avgcaptainrating
0,651fd9cb9781eb517f4de3bd,Link,640f45c9d7bf2958fb70ff74,577,h. 401 to 800,4.53
1,651fda1fbb70a9053e0a2d8a,Link,64ca87be858781c1fab0c437,544,h. 401 to 800,4.42


In [6]:
grouped_avg = df_base.groupby('captain_ltr')['avgcaptainrating'].transform('mean')

# Fill null values in 'value_column' with the corresponding group average
df_base['avgcaptainrating'] = df_base['avgcaptainrating'].fillna(grouped_avg)
df_base['avgcaptainrating'] = df_base['avgcaptainrating'].round(1)
df_base

,order_id,service_name,captain_id,lifetimerides,captain_ltr,avgcaptainrating
0,651fd9cb9781eb517f4de3bd,Link,640f45c9d7bf2958fb70ff74,577,h. 401 to 800,4.5
1,651fda1fbb70a9053e0a2d8a,Link,64ca87be858781c1fab0c437,544,h. 401 to 800,4.4
2,651fdab49a207c7b871ed03f,Auto,612f658dcb05bcab912dc5a8,1444,i. 800+,4.2
3,651fdafa7db73a4d2cb61326,Auto,646f2ffd885aa0300d90ee63,180,f. 101 to 200,4.2
4,651fdbeb48c8051c177f0be9,Link,5f7b49dc7d99a7913c5b0a44,252,g. 201 to 400,4.4
...,...,...,...,...,...,...
15025827,651a61090e36ff7664fa56d3,Auto,6333d7c862cc3b6590c713c9,3335,i. 800+,4.5
15025828,651a566d8c67782b7b0ff5b1,Auto,602eed94e32da95eabb5fed1,1544,i. 800+,4.3
15025829,651a56e1ce45113227a29d6c,Auto,61c01e019d88022789a568c2,554,h. 401 to 800,4.6
15025830,651a56e1ce45113227a29d6d,Link,63455e1923dd9670d408d319,2371,i. 800+,4.2


In [7]:
df_base.groupby(['service_name', 'avgcaptainrating']).agg({'order_id' : 'nunique', 'captain_id' : 'nunique'})

order_id  captain_id
service_name avgcaptainrating                      
Auto         1.0                    135          65
             1.2                      2           1
             1.3                      9           1
             1.7                      2           1
             1.8                      5           1
             2.0                     59          27
             2.2                     15           6
             2.3                    118          26
             2.4                     27           3
             2.5                     73          20
             2.6                     41          14
             2.7                     75          21
             2.8                    135          24
             2.9                     73          14
             3.0                    703         227
             3.1                    273          48
             3.2                    602         104
             3.3                    803         150
             3.4                   1578         227
             3.5                   2208         329
             3.6                   5073         607
             3.7                   7646         905
             3.8                  19042        1878
             3.9                  31152        2605
             4.0                  75609        6238
             4.1                 127559        9107
             4.2                 347423       21085
             4.3                 554721       28580
             4.4                1064255       49592
             4.5                1049127       47302
             4.6                1161478       48248
             4.7                 544592       19034
             4.8                 169085        7908
             4.9                  17805        2230
             5.0                  12955        3371
Link         1.0                    668         276
             1.2                      2           1
             1.3                      7           2
             1.5                     49          14
             1.7                     13           4
             1.8                      3           2
             2.0                    349         124
             2.2                     13           2
             2.3                    181          41
             2.4                     18           3
             2.5                    265          67
             2.6                     45           9
             2.7                    214          40
             2.8                    143          35
             2.9                    123          16
             3.0                   2994         795
             3.1                    354          43
             3.2                   1013         176
             3.3                   1887         262
             3.4                   2011         267
             3.5                   3210         549
             3.6                   4988         565
             3.7                   9769        1270
             3.8                  16352        1614
             3.9                  24394        1961
             4.0                  57590        5737
             4.1                  91894        6303
             4.2                 245096       15444
             4.3                 467586       24757
             4.4                1328189       56552
             4.5                2084212       75498
             4.6                3029838      106293
             4.7                1554071       58658
             4.8                 645847       33316
             4.9                 138633       11476
             5.0                 119353       22058

In [8]:
conditions = [
    (df_base['avgcaptainrating'] >= 1.0) & (df_base['avgcaptainrating'] < 2.0),
    (df_base['avgcaptainrating'] >= 2.0) & (df_base['avgcaptainrating'] < 3.0),
    (df_base['avgcaptainrating'] >= 3.0) & (df_base['avgcaptainrating'] <= 3.5),
    (df_base['avgcaptainrating'] >= 3.6) & (df_base['avgcaptainrating'] <= 3.7),
    (df_base['avgcaptainrating'] >= 3.8) & (df_base['avgcaptainrating'] <= 3.9),
]

values = [2.0, 3.0, 3.5, 3.7, 3.9]

df_base['rounded_avgcaptainrating'] = df_base['avgcaptainrating']

for condition, value in zip(conditions, values):
    df_base['rounded_avgcaptainrating'] = np.where(condition, value, df_base['rounded_avgcaptainrating'])                   

In [9]:
df_base.rounded_avgcaptainrating.unique()

array([4.5, 4.4, 4.2, 4.3, 4.1, 4.6, 4.7, 3.9, 4.8, 5. , 4.9, 4. , 3.7,
       3.5, 2. , 3. ])

In [10]:
df_base \
.groupby(['service_name', 'rounded_avgcaptainrating']) \
.agg({'order_id' : 'nunique', 'captain_id' : 'nunique'})         

order_id  captain_id
service_name rounded_avgcaptainrating                      
Auto         2.0                            153          69
             3.0                            616         155
             3.5                           6167        1085
             3.7                          12719        1512
             3.9                          50194        4483
             4.0                          75609        6238
             4.1                         127559        9107
             4.2                         347423       21085
             4.3                         554721       28580
             4.4                        1064255       49592
             4.5                        1049127       47302
             4.6                        1161478       48248
             4.7                         544592       19034
             4.8                         169085        7908
             4.9                          17805        2230
             5.0                          12955        3371
Link         2.0                            742         299
             3.0                           1351         337
             3.5                          11469        2092
             3.7                          14757        1835
             3.9                          40746        3575
             4.0                          57590        5737
             4.1                          91894        6303
             4.2                         245096       15444
             4.3                         467586       24757
             4.4                        1328189       56552
             4.5                        2084212       75498
             4.6                        3029838      106293
             4.7                        1554071       58658
             4.8                         645847       33316
             4.9                         138633       11476
             5.0                         119353       22058

##routeDIff

routeDIff = f"""

with orders_tbl as 
    ( SELECT
            orderdate,
            city,
            service_name,
            estimate_id, 
            estimate_ids,
            captain_id,
            fare_recalculated_reason reason,
            amount_breakup_prev_due_total,
            order_id,
            order_amount,
            fe_re_type,
            fe_re_diff_amount,
            distance_final_distance,
            u.id_array as fare_estimate_id
        FROM
            (
            SELECT   
                city_name as city,
                service_obj_service_name as service_name,
                order_id,
                captain_id,
                amount as order_amount,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                epoch,
                estimate_id,
                estimate_ids,
                CASE 
                WHEN  customer_obj_gender = '0' THEN 'Male'
                WHEN customer_obj_gender = '1' THEN 'Female'
                ELSE 'Unknown/Other' END
                gender,
                amount_breakup_prev_due_total,
                distance_final_distance,
                fare_recalculated_reason,
                fare_recalculated_type AS fe_re_type,
                fare_recalculated_diff_amount AS fe_re_diff_amount,
                cast(json_parse(estimate_ids) as array<varchar>) AS id_array
            FROM
                orders.order_logs_snapshot
            WHERE
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                AND order_status in ('dropped')
                AND spd_fraud_flag != True
                AND service_obj_service_name IN ('Link', 'Auto')
            ) as t
        CROSS JOIN UNNEST(t.id_array) AS u(id_array)),

fe_tbl as (
        SELECT
            city AS city,
            user_id as customer_id,
            fare_estimate_id,
            service_name,
            api_context,
            date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
            cast(final_amount AS double) AS fe_amount,
            ride_distance,
            epoch
        FROM
            pricing.fare_estimates_enriched
        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND service_name IN ('Link', 'Auto', 'Auto Pool', 'CabEconomy')
            
),
      

merged_tbl as (
    SELECT 
        *,
        CASE 
        WHEN amt_diff < -10 THEN '< -10'
        WHEN amt_diff >= -10 AND amt_diff < -7 THEN '-10 to -7'
        WHEN amt_diff >= -7 AND amt_diff < -5 THEN '-7 to -5'
        WHEN amt_diff >= -5 AND amt_diff < 0 THEN '-5 to 0'
        WHEN amt_diff > 0 AND amt_diff <= 5 THEN '0 to 5'
        WHEN amt_diff > 5 AND amt_diff <= 7 THEN '5 to 7'
        WHEN amt_diff > 7 AND amt_diff <= 10 THEN '7 to 10'
        WHEN amt_diff > 10 THEN '> 10'
        END AS amt_diff_bucket
    FROM
    (
    SELECT  
        o.orderdate,
        o.city,
        o.service_name,
        o.captain_id,
        o.order_id,
        o.estimate_id,
        estimate_ids,
        api_context,
        o.fare_estimate_id,
        o.reason,
        fe_amount,
        o.order_amount,
        o.order_amount -  fe_amount AS amt_diff,
        fe_re_type,
        fe_re_diff_amount,
        row_number() over (partition by o.estimate_id order by f.epoch asc)  as updated_seq 
    From orders_tbl o
    INNER JOIN fe_tbl f
    ON o.fare_estimate_id = f.fare_estimate_id
    AND o.service_name = f.service_name
    )

)
    
    SELECT
        orderdate,
        city,
        service_name,
        captain_id,
        order_id,
        estimate_id,
        fare_estimate_id,
        reason,
        fe_amount,
        order_amount,
        amt_diff,
        fe_re_type,
        fe_re_diff_amount,
        amt_diff_bucket
    FROM merged_tbl
    
    
    Where order_id in (SELECT order_id from merged_tbl where  amt_diff!=0)
    and order_id not in (SELECT order_id from merged_tbl where api_context = '/fare/editOrder')
    and updated_seq = 1
    and reason = 'routeDiff'
    
"""

df_routeDIff = pd.read_sql(routeDIff, connection)
df_routeDIff.to_csv('df_routeDIff_with_ltr.csv', index=False)
df_routeDIff

In [11]:
df_routeDIff = pd.read_csv('df_routeDIff_with_ltr.csv') 
df_routeDIff.head(2)

,orderdate,city,service_name,captain_id,order_id,estimate_id,fare_estimate_id,reason,fe_amount,order_amount,amt_diff,fe_re_type,fe_re_diff_amount,amt_diff_bucket
0,2023-10-15,Chennai,Link,638b311abaa3a5b058a48298,652bb6e5e27b914847feb3df,652bc32c544f0934b03c40b0,652bb698a4c61e3b013b1283,routeDiff,399.0,389.0,-10.0,decrease,10.0,-10 to -7
1,2023-10-08,Chennai,Link,5f4c9fb8f330ba75fa0c6b29,652284d37af97d5a950142d0,652289042537b4489f49bfec,652284c41be44517a90808fe,routeDiff,47.0,75.0,28.0,increase,28.0,> 10


## Analysis
### New vs. Old captains analysis with LTR bins

In [12]:
df_captain_analysis = pd.merge(df_base,
                               df_routeDIff,
                               how='left',
                               on=['order_id', 'captain_id']
                              )
df_captain_analysis.head(2)

,order_id,service_name_x,captain_id,lifetimerides,captain_ltr,avgcaptainrating,rounded_avgcaptainrating,orderdate,city,service_name_y,estimate_id,fare_estimate_id,reason,fe_amount,order_amount,amt_diff,fe_re_type,fe_re_diff_amount,amt_diff_bucket
0,651fd9cb9781eb517f4de3bd,Link,640f45c9d7bf2958fb70ff74,577,h. 401 to 800,4.5,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,651fda1fbb70a9053e0a2d8a,Link,64ca87be858781c1fab0c437,544,h. 401 to 800,4.4,4.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
## Taxi

df_agg = df_captain_analysis \
                            .groupby(['captain_ltr']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_captain_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_captain_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_captain_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_captain_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg['date_range'] = start_date + ' to -' + end_date
df_agg['routeDiff_orders %'] = df_agg['routeDiff_orders']*100.0/df_agg['total_net_orders']
df_agg['routeDiff_captain %'] = df_agg['routeDiff_captains']*100.0/df_agg['total_net_captains']
df_agg = df_agg[['date_range', 'captain_ltr', 
                 'total_net_captains', 'routeDiff_captains', 'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
df_agg.round(2)

,date_range,captain_ltr,total_net_captains,routeDiff_captains,routeDiff_captain %,total_net_orders,routeDiff_orders,routeDiff_orders %
0,20231002 to -20231015,a. 1 to 5,13535,4079,30.14,28080,5330,18.98
1,20231002 to -20231015,b. 6 to 10,12125,4968,40.97,49203,9159,18.61
2,20231002 to -20231015,c. 11 to 20,23151,10284,44.42,130967,24373,18.61
3,20231002 to -20231015,d. 21 to 50,61414,31049,50.56,489008,95714,19.57
4,20231002 to -20231015,e. 51 to 100,80965,45153,55.77,893035,175853,19.69
5,20231002 to -20231015,f. 101 to 200,114109,69315,60.74,1683455,337553,20.05
6,20231002 to -20231015,g. 201 to 400,133591,87754,65.69,2670398,538177,20.15
7,20231002 to -20231015,h. 401 to 800,118080,82486,69.86,3266311,647210,19.81
8,20231002 to -20231015,i. 800+,117074,84431,72.12,5815375,996953,17.14


In [14]:
## Taxi

df_agg = df_captain_analysis \
                            .groupby(['service_name_x', 'captain_ltr']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_captain_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_captain_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_captain_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_captain_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()
df_agg.rename(columns = {'service_name_x' : 'service_name'}, inplace=True)

df_agg['date_range'] = start_date + ' to -' + end_date
df_agg['routeDiff_orders %'] = df_agg['routeDiff_orders']*100.0/df_agg['total_net_orders']
df_agg['routeDiff_captain %'] = df_agg['routeDiff_captains']*100.0/df_agg['total_net_captains']
df_agg = df_agg[['date_range', 'service_name', 'captain_ltr', 
                 'total_net_captains', 'routeDiff_captains', 'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
df_agg.round(2)

,date_range,service_name,captain_ltr,total_net_captains,routeDiff_captains,routeDiff_captain %,total_net_orders,routeDiff_orders,routeDiff_orders %
0,20231002 to -20231015,Auto,a. 1 to 5,1722,594,34.49,2868,687,23.95
1,20231002 to -20231015,Auto,b. 6 to 10,1886,847,44.91,4944,1204,24.35
2,20231002 to -20231015,Auto,c. 11 to 20,4377,2139,48.87,15034,3541,23.55
3,20231002 to -20231015,Auto,d. 21 to 50,14864,8486,57.09,73137,17579,24.04
4,20231002 to -20231015,Auto,e. 51 to 100,24223,15845,65.41,174606,41833,23.96
5,20231002 to -20231015,Auto,f. 101 to 200,41499,29793,71.79,423184,100985,23.86
6,20231002 to -20231015,Auto,g. 201 to 400,56545,44467,78.64,837842,198389,23.68
7,20231002 to -20231015,Auto,h. 401 to 800,54481,45892,84.23,1219113,278663,22.86
8,20231002 to -20231015,Auto,i. 800+,50402,44646,88.58,2443730,420639,17.21
9,20231002 to -20231015,Link,a. 1 to 5,11813,3485,29.50,25212,4643,18.42


In [15]:
## Taxi

df_agg = df_captain_analysis \
                            .groupby(['rounded_avgcaptainrating']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_captain_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_captain_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_captain_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_captain_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg['date_range'] = start_date + ' to -' + end_date
df_agg['routeDiff_orders %'] = df_agg['routeDiff_orders']*100.0/df_agg['total_net_orders']
df_agg['routeDiff_captain %'] = df_agg['routeDiff_captains']*100.0/df_agg['total_net_captains']
df_agg = df_agg[['date_range', 'rounded_avgcaptainrating', 
                 'total_net_captains', 'routeDiff_captains', 'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
df_agg.round(2)

,date_range,rounded_avgcaptainrating,total_net_captains,routeDiff_captains,routeDiff_captain %,total_net_orders,routeDiff_orders,routeDiff_orders %
0,20231002 to -20231015,2.0,368,131,35.60,895,206,23.02
1,20231002 to -20231015,3.0,492,221,44.92,1967,440,22.37
2,20231002 to -20231015,3.5,3176,1456,45.84,17636,3712,21.05
3,20231002 to -20231015,3.7,3347,1809,54.05,27476,6021,21.91
4,20231002 to -20231015,3.9,8058,4840,60.06,90940,20536,22.58
5,20231002 to -20231015,4.0,11973,6987,58.36,133199,29703,22.30
6,20231002 to -20231015,4.1,15404,9901,64.28,219453,48480,22.09
7,20231002 to -20231015,4.2,36520,23869,65.36,592519,129216,21.81
8,20231002 to -20231015,4.3,53320,35334,66.27,1022307,213317,20.87
9,20231002 to -20231015,4.4,106120,69937,65.90,2392444,472169,19.74


<br><br><br>

## Segmentation

In [15]:
## Parameter 
start_date1 = '20231016'
end_date1 = '20231022'

##segmentation

segmentation = f"""

    with segment as (
    
    SELECT 
        captain_id,
        performance,
        consistency,
        pc_segment
    FROM 
    (
    SELECT 
        *,
        row_number() over(partition by captain_id order by epoch desc) rn
    FROM
        (
        SELECT   
            ols.yyyymmdd,
            ols.epoch,
            ols.service_obj_service_name service_name, 
            ols.captain_id,
            ols.order_id,
            
            CASE 
            WHEN ols.service_obj_service_name = 'Link' THEN b.performance_segment
            WHEN ols.service_obj_service_name = 'Auto' THEN a.performance
            ELSE 'Unknow/New'
            END AS performance,
            
            CASE 
            WHEN ols.service_obj_service_name = 'Link' THEN b.consistency
            WHEN ols.service_obj_service_name = 'Auto' THEN a.consistency
            ELSE 'Unknow/New'
            END AS consistency,
            
            CASE 
            WHEN ols.service_obj_service_name = 'Link' THEN b.performance_segment || '_' || b.consistency
            WHEN ols.service_obj_service_name = 'Auto' THEN a.pc_segment
            ELSE 'Unknow/New'
            END AS pc_segment
            
        FROM
            orders.order_logs_snapshot ols 
            
        LEFT JOIN
            datasets.captain_auto_pc_segments a
            ON ols.captain_id = a.captain_id
            AND ols.yyyymmdd = a.yyyymmdd
            AND ols.service_obj_service_name = a.service_name
            AND a.yyyymmdd >= '{start_date1}'
            AND a.yyyymmdd <= '{end_date1}'
        
        LEFT JOIN
            datasets.poc_segments_immutable_28_days b
            ON ols.captain_id = b.captain_id
            AND ols.yyyymmdd = b.yyyymmdd
            AND ols.service_obj_service_name = b.service_name
            AND ols.service_detail_id = b.servicedetailid
            AND b.yyyymmdd >= '{start_date1}'
            AND b.yyyymmdd <= '{end_date1}'
        
        WHERE
            ols.yyyymmdd >= '{start_date1}'
            AND ols.yyyymmdd <= '{end_date1}'
            AND order_status in ('dropped')
            AND spd_fraud_flag != True
            AND service_obj_service_name IN ('Link', 'Auto')
        )
    ) 
    WHERE 
        rn = 1
    ),    
    
    orders_tbl as 
    (
        SELECT   
            ols.order_id,
            ols.service_obj_service_name service_name, 
            ols.captain_id,
            segment.performance,
            segment.consistency,
            segment.pc_segment
        FROM
            orders.order_logs_snapshot ols
        
        LEFT JOIN segment
        ON ols.captain_id = segment.captain_id 
        
        WHERE
            ols.yyyymmdd >= '{start_date1}'
            AND ols.yyyymmdd <= '{end_date1}'
            AND order_status in ('dropped')
            AND spd_fraud_flag != True
            AND service_obj_service_name IN ('Link', 'Auto')
    ) 
    
    SELECT * FROM orders_tbl
        
"""

df_segmentation = pd.read_sql(segmentation, connection)
df_segmentation.to_csv('df_segmentation_{}_{}.csv'.format(start_date1, end_date1), index=False) 
df_segmentation

In [16]:
df_segmentation = pd.read_csv('df_segmentation_{}_{}.csv'.format(start_date1, end_date1)) 
df_segmentation.head(2)

,order_id,service_name,captain_id,performance,consistency,pc_segment
0,652c811ddd46a057208555d0,Link,64bf911317b096705a391bc4,UHP,daily,UHP_daily
1,652c816b8ca1ac1f45ed244d,Link,6014e1117143230bd4c4e7cb,HP,daily,HP_daily


##routeDIff

routeDIff1 = f"""

with orders_tbl as 
    ( SELECT
            orderdate,
            city,
            service_name,
            estimate_id, 
            estimate_ids,
            captain_id,
            fare_recalculated_reason reason,
            amount_breakup_prev_due_total,
            order_id,
            order_amount,
            fe_re_type,
            fe_re_diff_amount,
            distance_final_distance,
            u.id_array as fare_estimate_id
        FROM
            (
            SELECT   
                city_name as city,
                service_obj_service_name as service_name,
                order_id,
                captain_id,
                amount as order_amount,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                epoch,
                estimate_id,
                estimate_ids,
                CASE 
                WHEN  customer_obj_gender = '0' THEN 'Male'
                WHEN customer_obj_gender = '1' THEN 'Female'
                ELSE 'Unknown/Other' END
                gender,
                amount_breakup_prev_due_total,
                distance_final_distance,
                fare_recalculated_reason,
                fare_recalculated_type AS fe_re_type,
                fare_recalculated_diff_amount AS fe_re_diff_amount,
                cast(json_parse(estimate_ids) as array<varchar>) AS id_array
            FROM
                orders.order_logs_snapshot
            WHERE
                yyyymmdd >= '{start_date1}'
                AND yyyymmdd <= '{end_date1}'
                AND order_status in ('dropped')
                AND spd_fraud_flag != True
                AND service_obj_service_name IN ('Link', 'Auto')
            ) as t
        CROSS JOIN UNNEST(t.id_array) AS u(id_array)),

fe_tbl as (
        SELECT
            city AS city,
            user_id as customer_id,
            fare_estimate_id,
            service_name,
            api_context,
            date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
            cast(final_amount AS double) AS fe_amount,
            ride_distance,
            epoch
        FROM
            pricing.fare_estimates_enriched
        WHERE
            yyyymmdd >= '{start_date1}'
            AND yyyymmdd <= '{end_date1}'
            AND service_name IN ('Link', 'Auto', 'Auto Pool', 'CabEconomy')
            
),
      

merged_tbl as (
    SELECT 
        *,
        CASE 
        WHEN amt_diff < -10 THEN '< -10'
        WHEN amt_diff >= -10 AND amt_diff < -7 THEN '-10 to -7'
        WHEN amt_diff >= -7 AND amt_diff < -5 THEN '-7 to -5'
        WHEN amt_diff >= -5 AND amt_diff < 0 THEN '-5 to 0'
        WHEN amt_diff > 0 AND amt_diff <= 5 THEN '0 to 5'
        WHEN amt_diff > 5 AND amt_diff <= 7 THEN '5 to 7'
        WHEN amt_diff > 7 AND amt_diff <= 10 THEN '7 to 10'
        WHEN amt_diff > 10 THEN '> 10'
        END AS amt_diff_bucket
    FROM
    (
    SELECT  
        o.orderdate,
        o.city,
        o.service_name,
        o.captain_id,
        o.order_id,
        o.estimate_id,
        estimate_ids,
        api_context,
        o.fare_estimate_id,
        o.reason,
        fe_amount,
        o.order_amount,
        o.order_amount -  fe_amount AS amt_diff,
        fe_re_type,
        fe_re_diff_amount,
        row_number() over (partition by o.estimate_id order by f.epoch asc)  as updated_seq 
    From orders_tbl o
    INNER JOIN fe_tbl f
    ON o.fare_estimate_id = f.fare_estimate_id
    AND o.service_name = f.service_name
    )

)
    
    SELECT
        orderdate,
        city,
        service_name,
        captain_id,
        order_id,
        estimate_id,
        fare_estimate_id,
        reason,
        fe_amount,
        order_amount,
        amt_diff,
        fe_re_type,
        fe_re_diff_amount,
        amt_diff_bucket
    FROM merged_tbl
    
    
    Where order_id in (SELECT order_id from merged_tbl where  amt_diff!=0)
    and order_id not in (SELECT order_id from merged_tbl where api_context = '/fare/editOrder')
    and updated_seq = 1
    and reason = 'routeDiff'
    
"""

df_routeDIff1 = pd.read_sql(routeDIff1, connection)
df_routeDIff1.to_csv('df_routeDIff1_{}_{}.csv'.format(start_date1, end_date1), index=False) 
df_routeDIff1

In [17]:
df_routeDIff1 = pd.read_csv('df_routeDIff1_{}_{}.csv'.format(start_date1, end_date1)) 
df_routeDIff1.head(2)

,orderdate,city,service_name,captain_id,order_id,estimate_id,fare_estimate_id,reason,fe_amount,order_amount,amt_diff,fe_re_type,fe_re_diff_amount,amt_diff_bucket
0,2023-10-21,Chennai,Link,64808752c24bfecd58d830ba,65334e9a9abf3746b19e33e3,653351e5800b91ed59673d9e,65334e962a4f47448bc47ce7,routeDiff,34.0,32.0,-2.0,decrease,2.0,-5 to 0
1,2023-10-22,Hyderabad,Link,610020417b23c8f7c9a676a6,653454991686876329d14075,65345adf2bf378fac6382d81,653454873a9dc2b98cc75280,routeDiff,79.0,74.0,-5.0,decrease,5.0,-5 to 0


In [18]:
df_segment_analysis = pd.merge(df_segmentation,
                               df_routeDIff1,
                               how='left',
                               on=['captain_id','order_id']
                              )
df_segment_analysis.head(2)

,order_id,service_name_x,captain_id,performance,consistency,pc_segment,orderdate,city,service_name_y,estimate_id,fare_estimate_id,reason,fe_amount,order_amount,amt_diff,fe_re_type,fe_re_diff_amount,amt_diff_bucket
0,652c811ddd46a057208555d0,Link,64bf911317b096705a391bc4,UHP,daily,UHP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,652c816b8ca1ac1f45ed244d,Link,6014e1117143230bd4c4e7cb,HP,daily,HP_daily,2023-10-16,Jaipur,Link,652c8669a3f2d0e026b31850,652c814da3f2d0f9c2b2eb5f,routeDiff,65.0,56.0,-9.0,decrease,9.0,-10 to -7


In [19]:
## Taxi

df_agg = df_segment_analysis \
                            .groupby(['service_name_x', 'performance']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_segment_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_segment_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_segment_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_segment_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg.rename(columns = {'service_name_x' : 'service_name'}, inplace=True)

df_agg['date_range'] = start_date + ' to -' + end_date1
df_agg['routeDiff_orders %'] = df_agg['routeDiff_orders']*100.0/df_agg['total_net_orders']
df_agg['routeDiff_captain %'] = df_agg['routeDiff_captains']*100.0/df_agg['total_net_captains']
df_agg = df_agg[['date_range', 'service_name', 'performance', ## 'consistency', 'pc_segment', 
                 'total_net_captains', 'routeDiff_captains', 'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
df_agg[df_agg['total_net_captains'] > 10].round(2)

,date_range,service_name,performance,total_net_captains,routeDiff_captains,routeDiff_captain %,total_net_orders,routeDiff_orders,routeDiff_orders %
0,20231002 to -20231022,Auto,HP,13899,13012,93.62,660067,127558,19.33
1,20231002 to -20231022,Auto,LP,122281,72725,59.47,603866,150423,24.91
2,20231002 to -20231022,Auto,MP,74010,62713,84.74,1205466,281631,23.36
3,20231002 to -20231022,Auto,UHP,2968,2759,92.96,319383,28209,8.83
4,20231002 to -20231022,Link,HP,35270,23243,65.90,1323445,229497,17.34
5,20231002 to -20231022,Link,LP,134234,52000,38.74,749015,139408,18.61
6,20231002 to -20231022,Link,MP,138030,83391,60.42,2137787,402004,18.80
7,20231002 to -20231022,Link,UHP,11761,6835,58.12,809378,125984,15.57


Check for multiple weeks (min 3 weeks)

Auto
- UHP, HP -> 93% (Less routeDiff_orders %)
- ML-83%, LP-58% (High routeDiff_orders %)

Link
- UHP -> 59% (Less routeDiff_orders %)

- HP -> 67% (High routeDiff_orders %)
- ML-60% (High routeDiff_orders %)
- LP-58% (High routeDiff_orders %)

In [20]:
## Taxi

df_agg = df_segment_analysis \
                            .groupby(['service_name_x', 'consistency']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_segment_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_segment_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_segment_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_segment_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg.rename(columns = {'service_name_x' : 'service_name'}, inplace=True)

df_agg['date_range'] = start_date + ' to -' + end_date1
df_agg['routeDiff_orders %'] = df_agg['routeDiff_orders']*100.0/df_agg['total_net_orders']
df_agg['routeDiff_captain %'] = df_agg['routeDiff_captains']*100.0/df_agg['total_net_captains']
df_agg = df_agg[['date_range', 'service_name', 'consistency', ## 'consistency', 'pc_segment', 
                 'total_net_captains', 'routeDiff_captains', 'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
df_agg[df_agg['total_net_captains'] > 10].round(2)

,date_range,service_name,consistency,total_net_captains,routeDiff_captains,routeDiff_captain %,total_net_orders,routeDiff_orders,routeDiff_orders %
0,20231002 to -20231022,Auto,daily,88015,77365,87.90,1943522,404251,20.80
1,20231002 to -20231022,Auto,inter,60129,29715,49.42,282428,60417,21.39
3,20231002 to -20231022,Auto,intra,65008,44124,67.87,562765,123138,21.88
5,20231002 to -20231022,Link,daily,215426,121728,56.51,4448089,789159,17.74
6,20231002 to -20231022,Link,inter,25,13,52.00,277,67,24.19
7,20231002 to -20231022,Link,inter_week,34864,12440,35.68,115615,21553,18.64
9,20231002 to -20231022,Link,intra_week,68979,31288,45.36,455634,86114,18.90


Auto
- Daily > Intra > Inter

Link
- Daily > Inter > Intra week > Inter week

In [21]:

## Taxi

df_agg = df_segment_analysis \
                            .groupby(['service_name_x', 'pc_segment']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_segment_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_segment_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_segment_analysis['orderdate'].notna())
                                                                 &
                                                                 (df_segment_analysis['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg.rename(columns = {'service_name_x' : 'service_name'}, inplace=True)

df_agg['date_range'] = start_date + ' to -' + end_date1
df_agg['routeDiff_orders %'] = df_agg['routeDiff_orders']*100.0/df_agg['total_net_orders']
df_agg['routeDiff_captain %'] = df_agg['routeDiff_captains']*100.0/df_agg['total_net_captains']
df_agg = df_agg[['date_range', 'service_name', 'pc_segment', ## 'consistency', 'pc_segment', 
                 'total_net_captains', 'routeDiff_captains', 'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
df_agg[df_agg['total_net_captains'] > 10].round(2)

,date_range,service_name,pc_segment,total_net_captains,routeDiff_captains,routeDiff_captain %,total_net_orders,routeDiff_orders,routeDiff_orders %
0,20231002 to -20231022,Auto,HP_daily,9828,9496,96.62,520270,102951,19.79
1,20231002 to -20231022,Auto,HP_inter,1854,1548,83.50,51282,8815,17.19
2,20231002 to -20231022,Auto,HP_intra,2217,1968,88.77,88515,15792,17.84
3,20231002 to -20231022,Auto,LP_daily,30749,24658,80.19,268475,68174,25.39
4,20231002 to -20231022,Auto,LP_inter,46069,19719,42.80,114479,27862,24.34
6,20231002 to -20231022,Auto,LP_intra,45460,28345,62.35,220865,54375,24.62
8,20231002 to -20231022,Auto,MP_daily,45197,41064,90.86,884882,208914,23.61
9,20231002 to -20231022,Auto,MP_inter,11882,8190,68.93,99746,22330,22.39
11,20231002 to -20231022,Auto,MP_intra,16929,13458,79.50,220825,50385,22.82
12,20231002 to -20231022,Auto,UHP_daily,2241,2147,95.81,269895,24212,8.97


## Captain Rating + Segmentation

In [22]:
## Parameter 
start_date1 = '20231016'
end_date1 = '20231022'

##base

base1 = f"""

    with orders_tbl as 
        (
            SELECT   
                order_id,
                service_obj_service_name service_name, 
                captain_id,
                lifetimerides,
                CASE 
                WHEN lifetimerides BETWEEN 1 AND 5 THEN 'a. 1 to 5'
                WHEN lifetimerides BETWEEN 6 AND 10 THEN 'b. 6 to 10'
                WHEN lifetimerides BETWEEN 11 AND 20 THEN 'c. 11 to 20'
                WHEN lifetimerides BETWEEN 21 AND 50 THEN 'd. 21 to 50'
                WHEN lifetimerides > 50 AND lifetimerides <= 100 THEN 'e. 51 to 100'
                WHEN lifetimerides > 100 AND lifetimerides <= 200 THEN 'f. 101 to 200'
                WHEN lifetimerides > 200 AND lifetimerides <= 400 THEN 'g. 201 to 400'
                WHEN lifetimerides > 400 AND lifetimerides <= 800 THEN 'h. 401 to 800'
                WHEN lifetimerides > 800 THEN 'i. 800+'
                END captain_ltr,
                avgcaptainrating
            FROM
                orders.order_logs_snapshot
            
            INNER JOIN datasets.captain_single_view
            on captain_id = captainid 
            
            WHERE
                yyyymmdd >= '{start_date1}'
                AND yyyymmdd <= '{end_date1}'
    
                AND order_status in ('dropped')
                AND spd_fraud_flag != True
                AND service_obj_service_name IN ('Link', 'Auto')
        ) 
        
        SELECT * FROM orders_tbl
"""

df_base1 = pd.read_sql(base1, connection)
df_base1.to_csv('df_base1_with_ltr_{}_{}.csv'.format(start_date1, end_date1), index=False)
df_base1

##base

base1_lm = f"""

    with orders_tbl as 
        (
            SELECT   
                order_id,
                service_obj_service_name service_name, 
                captain_id,
                distance_final_distance last_mile
            FROM
                orders.order_logs_snapshot
            
            WHERE
                yyyymmdd >= '{start_date1}'
                AND yyyymmdd <= '{end_date1}'
    
                AND order_status in ('dropped')
                AND spd_fraud_flag != True
                AND service_obj_service_name IN ('Link', 'Auto')
        ) 
        
        SELECT * FROM orders_tbl
"""

df_base1_lm = pd.read_sql(base1_lm, connection)
df_base1_lm.to_csv('df_base1_with_lm_{}_{}.csv'.format(start_date1, end_date1), index=False)
df_base1_lm

In [23]:
df_base1_lm = pd.read_csv('df_base1_with_lm_{}_{}.csv'.format(start_date1, end_date1)) 
df_base1_lm.head(2)

,order_id,service_name,captain_id,last_mile
0,6532c6464c28820c53a60fed,Link,635baa95d58af72732db6a84,1.700
1,6532c6594033182c9ec341d8,Link,64e2e078fb75840616e1b1fc,5.142


In [24]:
df_base1 = pd.read_csv('df_base1_with_ltr_{}_{}.csv'.format(start_date1, end_date1)) 
df_base1.head(2)

,order_id,service_name,captain_id,lifetimerides,captain_ltr,avgcaptainrating
0,6534dfd1ccfcf22e7fd69ca3,Auto,5d2dc7786ed7674e61149fb2,1011,i. 800+,4.42
1,6534dfe64033182c9ec7dc9a,Link,63d3c3e5f17fa775d2ee41c3,1225,i. 800+,4.59


In [25]:
## merge lm info

df_base1 = pd.merge(df_base1,df_base1_lm, how='left', on=['order_id','service_name','captain_id'])

In [26]:
conditions = [
    (df_base1['last_mile'] >= 0.0) & (df_base1['last_mile'] < 2.0),
    (df_base1['last_mile'] >= 2.0) & (df_base1['last_mile'] < 5.0),
    (df_base1['last_mile'] >= 5.0) & (df_base1['last_mile'] < 8.0),
    (df_base1['last_mile'] >= 8.0) & (df_base1['last_mile'] < 12.0),
    (df_base1['last_mile'] >= 12.0),
    ]

values = ['below 2km', '2-5km', '5-8km', '8-12km', '12+ km']

df_base1['rounded_last_mile'] = df_base1['last_mile']

for condition, value in zip(conditions, values):
    df_base1['rounded_last_mile'] = np.where(condition, value, df_base1['rounded_last_mile'])   

In [27]:
grouped_avg1 = df_base1.groupby('captain_ltr')['avgcaptainrating'].transform('mean')

# Fill null values in 'value_column' with the corresponding group average
df_base1['avgcaptainrating'] = df_base1['avgcaptainrating'].fillna(grouped_avg1)
df_base1['avgcaptainrating'] = df_base1['avgcaptainrating'].round(1)
df_base1

,order_id,service_name,captain_id,lifetimerides,captain_ltr,avgcaptainrating,last_mile,rounded_last_mile
0,6534dfd1ccfcf22e7fd69ca3,Auto,5d2dc7786ed7674e61149fb2,1011,i. 800+,4.4,3.660,2-5km
1,6534dfe64033182c9ec7dc9a,Link,63d3c3e5f17fa775d2ee41c3,1225,i. 800+,4.6,5.980,5-8km
2,6534e2abe49af525e066a2ba,Link,5cb989fb54bc7263ff418c25,190,f. 101 to 200,4.6,1.530,below 2km
3,6534e2eafed63e17c49ca84e,Link,5d88beadaf7944215671fa16,275,g. 201 to 400,4.6,8.380,8-12km
4,6534e96c4033182c9ec7f4d5,Auto,6470c319703079827ea28013,704,h. 401 to 800,4.6,1.070,below 2km
...,...,...,...,...,...,...,...,...
7808376,653251e6dc9b7e3eae694451,Link,5d18f6493b752c45cf907e16,3843,i. 800+,4.7,4.185,2-5km
7808377,6532522833744d12a3cee65b,Auto,5d4a804d55fbf50d45e7651d,1049,i. 800+,4.5,9.460,8-12km
7808378,653253324033182c9ec20b98,Auto,629ae14c0fb5e9f1382d26ba,762,h. 401 to 800,4.6,1.240,below 2km
7808379,6532494047e18a6deb3f0870,Link,64966e1f4e8ea9b963388717,787,h. 401 to 800,4.6,5.070,5-8km


In [28]:
df_base1.groupby(['service_name', 'avgcaptainrating']).agg({'order_id' : 'nunique', 'captain_id' : 'nunique'})

order_id  captain_id
service_name avgcaptainrating                      
Auto         1.0                     91          50
             1.2                      3           1
             1.3                     13           4
             1.8                      1           1
             2.0                     24          11
             2.2                     13           6
             2.3                     39          11
             2.4                      3           2
             2.5                     27          16
             2.6                     29           7
             2.7                     40          17
             2.8                     62          20
             2.9                     25          12
             3.0                    427         156
             3.1                    235          43
             3.2                    285          74
             3.3                    605         124
             3.4                    770         172
             3.5                   1463         257
             3.6                   2771         492
             3.7                   4503         738
             3.8                  10070        1493
             3.9                  16591        2203
             4.0                  40498        5136
             4.1                  68327        7663
             4.2                 187956       17971
             4.3                 296165       24392
             4.4                 574412       42845
             4.5                 563744       40789
             4.6                 623932       41474
             4.7                 286950       16271
             4.8                  91206        6644
             4.9                  10394        1802
             5.0                   7108        2261
Link         1.0                    376         150
             1.3                      1           1
             1.5                     34           9
             1.9                      6           1
             2.0                    181          63
             2.2                     25           6
             2.3                    103          32
             2.4                     20           3
             2.5                    152          37
             2.6                     30           6
             2.7                     86          23
             2.8                     92          22
             2.9                     48           9
             3.0                   1569         465
             3.1                    134          23
             3.2                    528          91
             3.3                    691         157
             3.4                   1113         189
             3.5                   1828         333
             3.6                   2449         350
             3.7                   4599         759
             3.8                   8659        1117
             3.9                  12932        1431
             4.0                  29755        3784
             4.1                  45403        4455
             4.2                 124419       11226
             4.3                 235228       18262
             4.4                 669348       43210
             4.5                1042849       58744
             4.6                1543248       82861
             4.7                 818053       45819
             4.8                 344460       25059
             4.9                  70201        7984
             5.0                  60979       12615

In [29]:
df_base1_auto = df_base1[df_base1['service_name'].isin(['Auto'])]
df_base1_link = df_base1[df_base1['service_name'].isin(['Link'])]

In [30]:
conditions = [
    (df_base1_auto['avgcaptainrating'] >= 1.0) & (df_base1_auto['avgcaptainrating'] < 4.0),
    (df_base1_auto['avgcaptainrating'] >= 4.0) & (df_base1_auto['avgcaptainrating'] < 4.3),
    (df_base1_auto['avgcaptainrating'] >= 4.3) & (df_base1_auto['avgcaptainrating'] < 4.6),
    (df_base1_auto['avgcaptainrating'] >= 4.6) & (df_base1_auto['avgcaptainrating'] < 4.8),
    (df_base1_auto['avgcaptainrating'] >= 4.8) & (df_base1_auto['avgcaptainrating'] < 5.0),
    ]

values = [3.9, 4.3, 4.5, 4.7, 5.0]

df_base1_auto['rounded_avgcaptainrating'] = df_base1_auto['avgcaptainrating']

for condition, value in zip(conditions, values):
    df_base1_auto['rounded_avgcaptainrating'] = np.where(condition, value, df_base1_auto['rounded_avgcaptainrating'])   

In [31]:
conditions = [
    (df_base1_link['avgcaptainrating'] >= 1.0) & (df_base1_link['avgcaptainrating'] < 4.0),
    (df_base1_link['avgcaptainrating'] >= 4.0) & (df_base1_link['avgcaptainrating'] < 4.4),
    (df_base1_link['avgcaptainrating'] >= 4.4) & (df_base1_link['avgcaptainrating'] < 4.6),
    (df_base1_link['avgcaptainrating'] >= 4.6) & (df_base1_link['avgcaptainrating'] < 4.8),
    (df_base1_link['avgcaptainrating'] >= 4.8) & (df_base1_link['avgcaptainrating'] < 5.0),
    (df_base1_link['avgcaptainrating'] == 5.0),
    ]

values = [3.9, 4.3, 4.5, 4.7, 4.9, 5.0]

df_base1_link['rounded_avgcaptainrating'] = df_base1_link['avgcaptainrating']

for condition, value in zip(conditions, values):
    df_base1_link['rounded_avgcaptainrating'] = np.where(condition, value, df_base1_link['rounded_avgcaptainrating'])   

In [32]:
df_base1_auto.rounded_avgcaptainrating.unique()

array([4.5, 4.7, 4.3, 5. , 3.9])

In [33]:
df_base1_link.rounded_avgcaptainrating.unique()

array([4.7, 4.5, 4.9, 4.3, 3.9, 5. ])

In [34]:
frames = [df_base1_auto,df_base1_link]
df_basev1 = pd.concat(frames)

In [35]:
df_basev1.rounded_avgcaptainrating.unique()

array([4.5, 4.7, 4.3, 5. , 3.9, 4.9])

In [36]:
df_basev1.groupby(['service_name', 'rounded_avgcaptainrating']).agg({'order_id' : 'nunique', 'captain_id' : 'nunique'})

order_id  captain_id
service_name rounded_avgcaptainrating                      
Auto         3.9                          38090        5910
             4.3                         296781       30770
             4.5                        1434321      108026
             4.7                         910882       57745
             5.0                         108708       10707
Link         3.9                          35656        5277
             4.3                         434805       37727
             4.5                        1712197      101954
             4.7                        2361301      128680
             4.9                         414661       33043
             5.0                          60979       12615

In [37]:
df_basev1.groupby(['service_name', 'rounded_last_mile']).agg({'order_id' : 'nunique', 'captain_id' : 'nunique'})

order_id  captain_id
service_name rounded_last_mile                      
Auto         12+ km               435015      118781
             2-5km                946246      145176
             5-8km                611569      150153
             8-12km               453934      135208
             below 2km            342018       74790
Link         12+ km               382256      140143
             2-5km               2169794      267659
             5-8km               1133774      238037
             8-12km               630735      191479
             below 2km            703040      181692

In [38]:
df_basev1

,order_id,service_name,captain_id,lifetimerides,captain_ltr,avgcaptainrating,last_mile,rounded_last_mile,rounded_avgcaptainrating
0,6534dfd1ccfcf22e7fd69ca3,Auto,5d2dc7786ed7674e61149fb2,1011,i. 800+,4.4,3.660,2-5km,4.5
4,6534e96c4033182c9ec7f4d5,Auto,6470c319703079827ea28013,704,h. 401 to 800,4.6,1.070,below 2km,4.7
6,6534eadfc53bf00fc8a6f0b2,Auto,5f895ee0fb10b88d9e2471d0,2697,i. 800+,4.5,9.903,8-12km,4.5
7,6534deed1686876329d2907d,Auto,6187c5f9b281d73c9ac87c50,1459,i. 800+,4.5,1.770,below 2km,4.5
9,6534e0b98c1cc00bb7a2b78d,Auto,6177e11397513c3c18d5a9c6,1722,i. 800+,4.6,0.510,below 2km,4.7
...,...,...,...,...,...,...,...,...,...
7808373,65325079a20d2d0c2e9cd271,Link,64ace9dcc4d7796e3d85614f,835,i. 800+,4.7,4.240,2-5km,4.7
7808374,653250bda20d2d0c2e9cd34e,Link,64d8a4148103d9ac3b9ed5a5,244,g. 201 to 400,4.6,0.980,below 2km,4.7
7808375,653251cd33744d12a3cee542,Link,64004a9e8e212cc2c64bf041,228,g. 201 to 400,4.4,1.654,below 2km,4.5
7808376,653251e6dc9b7e3eae694451,Link,5d18f6493b752c45cf907e16,3843,i. 800+,4.7,4.185,2-5km,4.7


In [39]:
df_segment_analysis

,order_id,service_name_x,captain_id,performance,consistency,pc_segment,orderdate,city,service_name_y,estimate_id,fare_estimate_id,reason,fe_amount,order_amount,amt_diff,fe_re_type,fe_re_diff_amount,amt_diff_bucket
0,652c811ddd46a057208555d0,Link,64bf911317b096705a391bc4,UHP,daily,UHP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,652c816b8ca1ac1f45ed244d,Link,6014e1117143230bd4c4e7cb,HP,daily,HP_daily,2023-10-16,Jaipur,Link,652c8669a3f2d0e026b31850,652c814da3f2d0f9c2b2eb5f,routeDiff,65.0,56.0,-9.0,decrease,9.0,-10 to -7
2,652c7de947b95d6fddcfdde3,Auto,63c7796d87dc9022eddc759a,MP,daily,MP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,652c7b2a8ca1ac1f45ed1c73,Link,62239ce3f0b26be55061fda9,LP,daily,LP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,652c80d68ca1ac1f45ed238b,Auto,62a0db8317af9a5132512b10,LP,intra,LP_intra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7808405,65321336dc9b7e3eae6880c4,Link,61fb9130ebef824f2f2c432d,LP,daily,LP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7808406,65321458e47099469228881a,Link,633c38d7f5442149ff034fea,MP,daily,MP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7808407,65321491e4709946922888b1,Link,62e56411d8a2490105b085a1,HP,daily,HP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7808408,653214bb1686876329cbf28b,Auto,5f6dcbdf7d99a7734d561e10,MP,intra,MP_intra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
df_rating_segment = pd.merge(df_segment_analysis,
                             df_basev1,
                             how='left',
                             left_on = ['captain_id', 'order_id', 'service_name_x'],
                             right_on = ['captain_id', 'order_id', 'service_name']
                            )
df_rating_segment

,order_id,service_name_x,captain_id,performance,consistency,pc_segment,orderdate,city,service_name_y,estimate_id,fare_estimate_id,reason,fe_amount,order_amount,amt_diff,fe_re_type,fe_re_diff_amount,amt_diff_bucket,service_name,lifetimerides,captain_ltr,avgcaptainrating,last_mile,rounded_last_mile,rounded_avgcaptainrating
0,652c811ddd46a057208555d0,Link,64bf911317b096705a391bc4,UHP,daily,UHP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Link,932.0,i. 800+,4.6,5.790,5-8km,4.7
1,652c816b8ca1ac1f45ed244d,Link,6014e1117143230bd4c4e7cb,HP,daily,HP_daily,2023-10-16,Jaipur,Link,652c8669a3f2d0e026b31850,652c814da3f2d0f9c2b2eb5f,routeDiff,65.0,56.0,-9.0,decrease,9.0,-10 to -7,Link,986.0,i. 800+,4.5,4.876,2-5km,4.5
2,652c7de947b95d6fddcfdde3,Auto,63c7796d87dc9022eddc759a,MP,daily,MP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Auto,1404.0,i. 800+,4.4,2.890,2-5km,4.5
3,652c7b2a8ca1ac1f45ed1c73,Link,62239ce3f0b26be55061fda9,LP,daily,LP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Link,501.0,h. 401 to 800,4.8,1.340,below 2km,4.9
4,652c80d68ca1ac1f45ed238b,Auto,62a0db8317af9a5132512b10,LP,intra,LP_intra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Auto,175.0,f. 101 to 200,4.7,3.850,2-5km,4.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7808405,65321336dc9b7e3eae6880c4,Link,61fb9130ebef824f2f2c432d,LP,daily,LP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Link,330.0,g. 201 to 400,4.6,0.950,below 2km,4.7
7808406,65321458e47099469228881a,Link,633c38d7f5442149ff034fea,MP,daily,MP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Link,280.0,g. 201 to 400,4.5,3.200,2-5km,4.5
7808407,65321491e4709946922888b1,Link,62e56411d8a2490105b085a1,HP,daily,HP_daily,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Link,430.0,h. 401 to 800,4.4,1.510,below 2km,4.5
7808408,653214bb1686876329cbf28b,Auto,5f6dcbdf7d99a7734d561e10,MP,intra,MP_intra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Auto,547.0,h. 401 to 800,4.2,13.960,12+ km,4.3


In [41]:
## Taxi

df_agg2 = df_rating_segment \
                            .groupby(['service_name_x', 'performance']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg2['date_range'] = start_date1 + ' to -' + end_date1
df_agg2['routeDiff_orders %'] = df_agg2['routeDiff_orders']*100.0/df_agg2['total_net_orders']
df_agg2['routeDiff_captain %'] = df_agg2['routeDiff_captains']*100.0/df_agg2['total_net_captains']
df_agg2 = df_agg2[['service_name_x', 'performance',
                 'total_net_captains', 'routeDiff_captains', ##'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
print(start_date1, end_date1)
df_agg2.round(2)

20231016 20231022


,service_name_x,performance,total_net_captains,routeDiff_captains,total_net_orders,routeDiff_orders,routeDiff_orders %
0,Auto,HP,13899,13012,660067,127558,19.33
1,Auto,LP,122281,72725,603866,150423,24.91
2,Auto,MP,74010,62713,1205466,281631,23.36
3,Auto,UHP,2968,2759,319383,28209,8.83
4,Link,HP,35270,23243,1323445,229497,17.34
5,Link,LP,134234,52000,749015,139408,18.61
6,Link,MP,138030,83391,2137787,402004,18.80
7,Link,UHP,11761,6835,809378,125984,15.57
8,Link,ZP,3,2,3,2,66.67


In [42]:
## Taxi

df_agg2 = df_rating_segment \
                            .groupby(['service_name_x', 'performance','rounded_last_mile']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg2['date_range'] = start_date1 + ' to -' + end_date1
df_agg2['routeDiff_orders %'] = df_agg2['routeDiff_orders']*100.0/df_agg2['total_net_orders']
df_agg2['routeDiff_captain %'] = df_agg2['routeDiff_captains']*100.0/df_agg2['total_net_captains']
df_agg2 = df_agg2[['service_name_x', 'performance', 'rounded_last_mile',
                 'total_net_captains', 'routeDiff_captains', ##'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
print(start_date1, end_date1)
df_agg2.round(2)

20231016 20231022


,service_name_x,performance,rounded_last_mile,total_net_captains,routeDiff_captains,total_net_orders,routeDiff_orders,routeDiff_orders %
0,Auto,HP,12+ km,10075,7672,64243,24729,38.49
1,Auto,HP,2-5km,13476,10368,257645,34698,13.47
2,Auto,HP,5-8km,13290,10512,148572,32672,21.99
3,Auto,HP,8-12km,12115,9266,94410,28504,30.19
4,Auto,HP,below 2km,11740,4196,95197,6955,7.31
5,Auto,LP,12+ km,62429,35270,161582,57890,35.83
6,Auto,LP,2-5km,65085,18309,141747,22025,15.54
7,Auto,LP,5-8km,70635,24794,139732,30684,21.96
8,Auto,LP,8-12km,65291,27992,127702,36267,28.40
9,Auto,LP,below 2km,22387,3343,33103,3557,10.75


Auto
- MP/LP > HP > UHP


Link 
- MP/LP > HP > UHP

In [43]:
## Taxi

df_agg2 = df_rating_segment \
                            .groupby(['service_name_x', 'rounded_avgcaptainrating']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg2['date_range'] = start_date1 + ' to -' + end_date1
df_agg2['routeDiff_orders %'] = df_agg2['routeDiff_orders']*100.0/df_agg2['total_net_orders']
df_agg2['routeDiff_captain %'] = df_agg2['routeDiff_captains']*100.0/df_agg2['total_net_captains']
df_agg2 = df_agg2[['service_name_x', 'rounded_avgcaptainrating',
                 'total_net_captains', 'routeDiff_captains', ##'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
print(start_date1, end_date1)
df_agg2.round(2)

20231016 20231022


,service_name_x,rounded_avgcaptainrating,total_net_captains,routeDiff_captains,total_net_orders,routeDiff_orders,routeDiff_orders %
0,Auto,3.9,5910,3903,38090,11357,29.82
1,Auto,4.3,30770,22494,296781,82154,27.68
2,Auto,4.5,108026,79347,1434321,333723,23.27
3,Auto,4.7,57745,39363,910882,145298,15.95
4,Auto,5.0,10707,6102,108708,15289,14.06
5,Link,3.9,5277,1963,35656,5248,14.72
6,Link,4.3,37727,16555,434805,67109,15.43
7,Link,4.5,101954,54288,1712197,305639,17.85
8,Link,4.7,128680,71948,2361301,439280,18.60
9,Link,4.9,33043,15969,414661,69506,16.76


Auto
- Below 4 > Middle > Above 4.3

Link
- Below 4 < Middle < Above 4.3

In [44]:
## Taxi

df_agg2 = df_rating_segment \
                            .groupby(['service_name_x', 'rounded_avgcaptainrating', 'performance']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg2['date_range'] = start_date1 + ' to -' + end_date1
df_agg2['routeDiff_orders %'] = df_agg2['routeDiff_orders']*100.0/df_agg2['total_net_orders']
df_agg2['routeDiff_captain %'] = df_agg2['routeDiff_captains']*100.0/df_agg2['total_net_captains']
df_agg2 = df_agg2[['service_name_x', 'rounded_avgcaptainrating', 'performance',
                 'total_net_captains', 'routeDiff_captains', ##'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]
print(start_date1, end_date1)
df_agg2[df_agg2['total_net_captains'] > 1].round(2)

20231016 20231022


,service_name_x,rounded_avgcaptainrating,performance,total_net_captains,routeDiff_captains,total_net_orders,routeDiff_orders,routeDiff_orders %
0,Auto,3.9,HP,61,59,2366,613,25.91
1,Auto,3.9,LP,4411,2608,17541,5397,30.77
2,Auto,3.9,MP,1437,1235,18168,5342,29.40
4,Auto,4.3,HP,858,840,37522,9534,25.41
5,Auto,4.3,LP,19609,12560,96511,27848,28.85
6,Auto,4.3,MP,10263,9055,159207,44216,27.77
7,Auto,4.3,UHP,40,39,3541,556,15.70
8,Auto,4.5,HP,7425,7106,356492,77772,21.82
9,Auto,4.5,LP,60109,36954,313235,78663,25.11
10,Auto,4.5,MP,39623,34457,680464,164646,24.20


Auto

- Below 4
    - MP/LP -> 30%
    - HP -> 25%
   
- Middle 
    - MP/HP -> 28
    - HP -> 25%
    
- Above 4.3
    - MP/LP -> ~23.5%
    - HP -> 18%
    
Link

- Below 4
    - MP -> 16.7%
    - LP -> 13.8%
    - HP -> 12%
   
- Middle 
    - MP -> 16%
    - LP -> 14.5%
    - HP -> 15%
    
- Above 4.3
    - MP -> 18%
    - LP -> 19%
    - HP -> 17%

In [48]:
## Taxi

df_agg2 = df_rating_segment \
                            .groupby(['service_name_x', 'performance', 'rounded_avgcaptainrating', 'rounded_last_mile']) \
                            .agg(total_net_orders = pd.NamedAgg('order_id', 'nunique'),
                                 total_net_captains = pd.NamedAgg('captain_id', 'nunique'),
                                 routeDiff_orders = ('order_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique()),
                                 routeDiff_captains = ('captain_id', 
                                                     lambda x: x[(df_rating_segment['orderdate'].notna())
                                                                 &
                                                                 (df_rating_segment['orderdate'] != '')
                                                             ] \
                                                  .nunique())
                                ) \
                            .reset_index()

df_agg2['date_range'] = start_date1 + ' to -' + end_date1
df_agg2['routeDiff_orders %'] = df_agg2['routeDiff_orders']*100.0/df_agg2['total_net_orders']
df_agg2['routeDiff_captain %'] = df_agg2['routeDiff_captains']*100.0/df_agg2['total_net_captains']
df_agg2 = df_agg2[['service_name_x', 'performance', 'rounded_avgcaptainrating', 'rounded_last_mile',
                 'total_net_captains', 'routeDiff_captains', ##'routeDiff_captain %',
                 'total_net_orders', 'routeDiff_orders', 'routeDiff_orders %'
                ]]

df_agg2.rename(columns = {'rounded_avgcaptainrating' : 'captainrating', 
                          'rounded_last_mile' : 'last_mile',
                          'service_name_x' : 'service_name'
                         }, inplace = True)


print(start_date1, end_date1)
df_agg2 = df_agg2[df_agg2['total_net_captains'] > 1].round(2)
df_agg2

20231016 20231022


,service_name,performance,captainrating,last_mile,total_net_captains,routeDiff_captains,total_net_orders,routeDiff_orders,routeDiff_orders %
0,Auto,HP,3.9,12+ km,52,44,405,167,41.23
1,Auto,HP,3.9,2-5km,56,40,708,122,17.23
2,Auto,HP,3.9,5-8km,59,49,566,157,27.74
3,Auto,HP,3.9,8-12km,59,50,439,140,31.89
4,Auto,HP,3.9,below 2km,44,18,248,27,10.89
5,Auto,HP,4.3,12+ km,713,598,5924,2559,43.20
6,Auto,HP,4.3,2-5km,829,631,11550,1863,16.13
7,Auto,HP,4.3,5-8km,841,711,9003,2288,25.41
8,Auto,HP,4.3,8-12km,805,686,7092,2407,33.94
9,Auto,HP,4.3,below 2km,666,254,3953,417,10.55


In [51]:
df_agg2.to_clipboard(index=False)